# Imports

In [3]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device: ' + str(device))

Using device: cuda


# Data

In [4]:
# Since params inside ResNet18 is based of these values, we should use them and not the ones for oxford-IIIT
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

# Again 224 is needed since its what ResNet18 wants as input
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# For binary (0,1)
train_set_binary = OxfordIIITPet(root='data', split='trainval', target_types='binary-category', transform=train_transform, download=True)
test_set_binary  = OxfordIIITPet(root='data', split='test', target_types='binary-category', transform=test_transform,  download=True)

train_loader_binary = DataLoader(train_set_binary, batch_size=50, shuffle=True,  num_workers=4) # Maybe batch size or/and num_workers should change value
test_loader_binary  = DataLoader(test_set_binary, batch_size=50, shuffle=False, num_workers=4)

# For full 37 (0-36)
train_set_full = OxfordIIITPet(root='data', split='trainval', target_types='category', transform=train_transform, download=True)
test_set_full  = OxfordIIITPet(root='data', split='test', target_types='category', transform=test_transform,  download=True)

train_loader_full = DataLoader(train_set_full, batch_size=50, shuffle=True,  num_workers=4)
test_loader_full  = DataLoader(test_set_full, batch_size=50, shuffle=False, num_workers=4)



print(f'Train size for binary: {len(train_set_binary)}, Test size: {len(test_set_binary)}')
print(f'Train size for full: {len(train_set_full)}, Test size: {len(test_set_full)}')

Train size for binary: 3680, Test size: 3669
Train size for full: 3680, Test size: 3669


# Training for Binary Classification

In [7]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2).to(device)
model = model.to(device)


In [8]:
# Training for binary classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_binary:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_binary)
    train_acc  = correct / len(train_set_binary)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_binary:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0 = cat, 1 = dog for binary
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_binary)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')

Epoch [01/2]  Loss: 0.2541  Train Acc: 0.9071  Test Acc: 0.9730
Epoch [02/2]  Loss: 0.0926  Train Acc: 0.9747  Test Acc: 0.9790


In [ ]:
# print('Final test accuracy for binary classification: ' + str(test_acc*100) + '%') # I get 97.7% after 2 epochs in vscode (~3min per epoch, takes a lot longer in colab)

# Training for Full Classification

In [ ]:
"""model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)"""

"model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?\n\n# Matching the tutorial\nfor param in model.parameters():\n    param.requires_grad = False\n\nnum_features = model.fc.in_features\nmodel.fc = nn.Linear(num_features, 37).to(device)"

In [ ]:
"""# Training for full classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_full:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0-36 depending on breed
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_full)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')"""

"# Training for full classification\ncriterion = nn.CrossEntropyLoss()\noptimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%\n\nnum_epochs = 2\n\nfor epoch in range(num_epochs):\n    model.train()\n    running_loss = 0.0\n    correct = 0\n\n    for images, labels in train_loader_full:\n        images, labels = images.to(device), labels.to(device)\n\n        optimizer.zero_grad()\n        outputs = model(images)\n        loss = criterion(outputs, labels)\n        loss.backward()\n        optimizer.step()\n\n        running_loss += loss.item() * images.size(0)\n        correct += (outputs.argmax(1) == labels).sum().item()\n\n    train_loss = running_loss / len(train_set_full)\n    train_acc  = correct / len(train_set_full)\n\n\n    model.eval()\n    correct = 0\n    with torch.no_grad():\n        for images, labels in test_loader_full:\n            images, labels = images.to(device), labels.to(device)\n            outputs = model(imag

In [ ]:
print('Final test accuracy for full classification: ' + str(test_acc*100) + '%') # I get 84.4% after 2 epochs in vscode (~3min per epoch, takes a lot longer in colab)

NameError: name 'test_acc' is not defined

# Strategy 1

Unfreeze 1 layer

In [ ]:
"""model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

trainable_blocks = []

for name, module in model.named_children():
    has_params = any(p.requires_grad is not None for p in module.parameters())

    if has_params:
        trainable_blocks.append((name, module))

last_block_name, last_block = trainable_blocks[-2]

for param in last_block.parameters():
    param.requires_grad = True

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
"""

"model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?\n\n# Matching the tutorial\nfor param in model.parameters():\n    param.requires_grad = False\n\ntrainable_blocks = []\n\nfor name, module in model.named_children():\n    has_params = any(p.requires_grad is not None for p in module.parameters())\n\n    if has_params:\n        trainable_blocks.append((name, module))\n\nlast_block_name, last_block = trainable_blocks[-2]\n\nfor param in last_block.parameters():\n    param.requires_grad = True\n\nnum_features = model.fc.in_features\nmodel.fc = nn.Linear(num_features, 37).to(device)\n"

In [ ]:
"""# Training for full classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(list(model.fc.parameters()) + list(last_block.parameters()), lr=1e-4) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_full:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0-36 depending on breed
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_full)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')"""

"# Training for full classification\ncriterion = nn.CrossEntropyLoss()\noptimizer = torch.optim.Adam(list(model.fc.parameters()) + list(last_block.parameters()), lr=1e-4) # Maybe lr should change for greater than 99%\n\nnum_epochs = 2\n\nfor epoch in range(num_epochs):\n    model.train()\n    running_loss = 0.0\n    correct = 0\n\n    for images, labels in train_loader_full:\n        images, labels = images.to(device), labels.to(device)\n\n        optimizer.zero_grad()\n        outputs = model(images)\n        loss = criterion(outputs, labels)\n        loss.backward()\n        optimizer.step()\n\n        running_loss += loss.item() * images.size(0)\n        correct += (outputs.argmax(1) == labels).sum().item()\n\n    train_loss = running_loss / len(train_set_full)\n    train_acc  = correct / len(train_set_full)\n\n\n    model.eval()\n    correct = 0\n    with torch.no_grad():\n        for images, labels in test_loader_full:\n            images, labels = images.to(device), labels.to(dev

In [ ]:
print('Final test accuracy for full classification with one unefreezed layer: ' + str(test_acc*100) + '%') # I get 86.9% after 2 epochs in vscode (~7 min total, takes a lot longer in colab)

Final test accuracy for full classification with one unefreezed layer: 87.4897792313982%


2 last layers unfrozen

In [ ]:
"""model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

trainable_blocks = []

for name, module in model.named_children():
    has_params = any(p.requires_grad is not None for p in module.parameters())

    if has_params:
        trainable_blocks.append((name, module))

l = 2 # number of layers to unfreeze

last_blocks = trainable_blocks[-(l+1):-1]



for name, block in last_blocks:
    for param in block.parameters():
        param.requires_grad = True

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
model = model.to(device)

"""

"model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?\n\n# Matching the tutorial\nfor param in model.parameters():\n    param.requires_grad = False\n\ntrainable_blocks = []\n\nfor name, module in model.named_children():\n    has_params = any(p.requires_grad is not None for p in module.parameters())\n\n    if has_params:\n        trainable_blocks.append((name, module))\n\nl = 2 # number of layers to unfreeze\n\nlast_blocks = trainable_blocks[-(l+1):-1]\n\n\n\nfor name, block in last_blocks:\n    for param in block.parameters():\n        param.requires_grad = True\n\nnum_features = model.fc.in_features\nmodel.fc = nn.Linear(num_features, 37).to(device)\nmodel = model.to(device)\n\n"

In [ ]:
"""# Training for full classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_full:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0-36 depending on breed
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_full)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')"""

"# Training for full classification\ncriterion = nn.CrossEntropyLoss()\noptimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4) # Maybe lr should change for greater than 99%\n\nnum_epochs = 2\n\nfor epoch in range(num_epochs):\n    model.train()\n    running_loss = 0.0\n    correct = 0\n\n    for images, labels in train_loader_full:\n        images, labels = images.to(device), labels.to(device)\n\n        optimizer.zero_grad()\n        outputs = model(images)\n        loss = criterion(outputs, labels)\n        loss.backward()\n        optimizer.step()\n\n        running_loss += loss.item() * images.size(0)\n        correct += (outputs.argmax(1) == labels).sum().item()\n\n    train_loss = running_loss / len(train_set_full)\n    train_acc  = correct / len(train_set_full)\n\n\n    model.eval()\n    correct = 0\n    with torch.no_grad():\n        for images, labels in test_loader_full:\n            images, labels = images.to(device), labels.to(device)\n

In [ ]:
print('Final test accuracy for full classification with one unefreezed layer: ' + str(test_acc*100) + '%') # I get 88.6% after 2 epochs in vscode (~8 min total, takes a lot longer in colab)

Final test accuracy for full classification with one unefreezed layer: 87.4897792313982%


3 last layers unfrozen

In [ ]:
"""model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

trainable_blocks = []

for name, module in model.named_children():
    has_params = any(p.requires_grad is not None for p in module.parameters())

    if has_params:
        trainable_blocks.append((name, module))

l = 3 # number of layers to unfreeze

last_blocks = trainable_blocks[-(l+1):-1]



for name, block in last_blocks:
    for param in block.parameters():
        param.requires_grad = True

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
model = model.to(device)

"""

"model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?\n\n# Matching the tutorial\nfor param in model.parameters():\n    param.requires_grad = False\n\ntrainable_blocks = []\n\nfor name, module in model.named_children():\n    has_params = any(p.requires_grad is not None for p in module.parameters())\n\n    if has_params:\n        trainable_blocks.append((name, module))\n\nl = 3 # number of layers to unfreeze\n\nlast_blocks = trainable_blocks[-(l+1):-1]\n\n\n\nfor name, block in last_blocks:\n    for param in block.parameters():\n        param.requires_grad = True\n\nnum_features = model.fc.in_features\nmodel.fc = nn.Linear(num_features, 37).to(device)\nmodel = model.to(device)\n\n"

In [ ]:
"""# Training for full classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_full:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0-36 depending on breed
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_full)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')"""

"# Training for full classification\ncriterion = nn.CrossEntropyLoss()\noptimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4) # Maybe lr should change for greater than 99%\n\nnum_epochs = 2\n\nfor epoch in range(num_epochs):\n    model.train()\n    running_loss = 0.0\n    correct = 0\n\n    for images, labels in train_loader_full:\n        images, labels = images.to(device), labels.to(device)\n\n        optimizer.zero_grad()\n        outputs = model(images)\n        loss = criterion(outputs, labels)\n        loss.backward()\n        optimizer.step()\n\n        running_loss += loss.item() * images.size(0)\n        correct += (outputs.argmax(1) == labels).sum().item()\n\n    train_loss = running_loss / len(train_set_full)\n    train_acc  = correct / len(train_set_full)\n\n\n    model.eval()\n    correct = 0\n    with torch.no_grad():\n        for images, labels in test_loader_full:\n            images, labels = images.to(device), labels.to(device)\n

In [ ]:
print('Final test accuracy for full classification with one unefreezed layer: ' + str(test_acc*100) + '%') # I get 87.8% after 2 epochs in vscode (~10 min total, takes a lot longer in colab)

Final test accuracy for full classification with one unefreezed layer: 87.4897792313982%


# Strategy 2

In [ ]:

def unfreeze_last_blocks(trainable_blocks, l):

    if l == 0:
        print('Only fc is trainable')
        return

    last_blocks = trainable_blocks[-(l+1):-1]

    for name, block in last_blocks:
        for param in block.parameters():
            param.requires_grad = True

    print('Unfrozen blocks:', [name for name, _ in last_blocks])

In [ ]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

trainable_blocks = []

for name, module in model.named_children():
    has_params = any(p.requires_grad is not None for p in module.parameters())

    if has_params:
        trainable_blocks.append((name, module))

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
model = model.to(device)


In [ ]:
l = 2 # choose amount of unfrozen layers
stages = list(range(1,l+1))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


for stage in stages:
    criterion = nn.CrossEntropyLoss()
    unfreeze_last_blocks(trainable_blocks, stage)

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0

        for images, labels in train_loader_full:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()

        train_loss = running_loss / len(train_set_full)
        train_acc  = correct / len(train_set_full)



model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader_full:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        # 0-36 depending on breed
        correct += (outputs.argmax(1) == labels).sum().item()

test_acc = correct / len(test_set_full)

Unfrozen blocks: ['layer4']
Unfrozen blocks: ['layer3', 'layer4']


In [ ]:
print('Final test accuracy for full classification with one unefreezed layer: ' + str(test_acc*100) + '%') # I get 90.0% doing gradual unfreezing with two layers, 2 epochs in vscode (~11 min total, takes a lot longer in colab)

Final test accuracy for full classification with one unefreezed layer: 89.97001907876806%
